# Search-strategy comparison on the AI4I 2020 dataset

Analysis of three search arms on identical splits: Optuna TPE over a
shared scikit-learn space, symbolic regression with PySR, and a
pydantic-graph agent loop (glm-5.3-flash) that proposes configurations
from the trial history. The metric is mean average precision over a
stratified 5-fold cross-validation on the 8000-row search part. Each
arm's best configuration is scored once on the 2000-row holdout.

Arms 2 and 3 are single runs; the Optuna arm repeats with five seeds.
See results/environment.json for interpreter, package, CPU, and Julia
details.

In [1]:
import json
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd

from symbolic_optimization.trials import load_records

RESULTS = next(p for p in (Path("results"), Path("../results")) if p.is_dir())

rows = []
for path in sorted(RESULTS.glob("*.jsonl")):
    if path.stem == "baselines":
        continue
    for record in load_records(path):
        rows.append(
            {
                "log": path.stem,
                "arm": record.arm,
                "trial_index": record.trial_index,
                "mean_score": record.mean_score,
                "std_score": record.std_score,
                "wall_clock_s": record.wall_clock_s,
                "cumulative_s": record.cumulative_s,
                "params": record.params,
                "payload": record.payload,
            }
        )
trials = pd.DataFrame(rows)
baselines = pd.DataFrame(
    [
        {"arm": r.arm, "mean_score": r.mean_score}
        for r in load_records(RESULTS / "baselines.jsonl")
    ]
)
holdout = pd.DataFrame(json.loads((RESULTS / "holdout.json").read_text()))
environment = json.loads((RESULTS / "environment.json").read_text())
trials.groupby("log").agg(
    n=("trial_index", "size"),
    best=("mean_score", "max"),
    minutes=("cumulative_s", lambda s: s.max() / 60),
)

,n,best,minutes
log,,,
agent,50,0.833356,63.942771
optuna_seed0,50,0.826507,4.693756
optuna_seed1,50,0.815673,2.520081
optuna_seed2,50,0.825799,4.991306
optuna_seed3,50,0.824174,5.292037
optuna_seed4,50,0.830587,4.417850
pysr,41,0.545499,1.279691


## Loop structure of the agent arm

The agent arm runs a pydantic-graph state machine. The mermaid source
below documents the executed graph.

In [2]:
from symbolic_optimization.track_agent import build_graph

print(build_graph().render())

stateDiagram-v2
  start
  Propose
  state decision <<choice>>
  Evaluate
  state decision_2 <<choice>>
  Decide
  state decision_3 <<choice>>

  [*] --> start
  start --> Propose
  Propose --> decision
  decision --> Evaluate
  decision --> [*]
  Evaluate --> decision_2
  decision_2 --> [*]
  decision_2 --> Decide
  Decide --> decision_3
  decision_3 --> Propose
  decision_3 --> [*]


/home/parvis/Projects/Development/symbolic-optimization/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Best-so-far score by evaluation index

Cumulative maximum of the mean cross-validation score. For the PySR
arm, one evaluation is one hall-of-fame candidate scored on a
validation fold, so its evaluation counter advances faster per unit of
compute than the parametric arms. Dashed rules mark the two default
configurations and the ground-truth rule classifier.

In [3]:
best = trials.sort_values(["log", "trial_index"]).assign(
    best_score=lambda d: d.groupby("log")["mean_score"].cummax()
)
optuna = best[best["log"].str.startswith("optuna")]
band = (
    optuna.groupby("trial_index")["best_score"]
    .agg(median="median", low="min", high="max")
    .reset_index()
)
lines = pd.concat(
    [
        band.assign(label="optuna (median of 5 seeds)").rename(columns={"median": "y"})[
            ["trial_index", "y", "label"]
        ],
        best[best["log"] == "agent"].assign(label="agent", y=lambda d: d["best_score"])[
            ["trial_index", "y", "label"]
        ],
        best[best["log"] == "pysr"].assign(label="pysr", y=lambda d: d["best_score"])[
            ["trial_index", "y", "label"]
        ],
    ]
)
area = (
    alt.Chart(band)
    .mark_area(opacity=0.18)
    .encode(
        x=alt.X("trial_index", title="Evaluation index"),
        y=alt.Y("low", title="Best mean average precision so far"),
        y2="high",
    )
)
line = (
    alt.Chart(lines)
    .mark_line()
    .encode(x="trial_index", y="y", color=alt.Color("label", title="arm"))
)
rules = (
    alt.Chart(baselines)
    .mark_rule(strokeDash=[3, 3])
    .encode(
        y="mean_score",
        color=alt.Color("arm", title="reference", scale=alt.Scale(scheme="set2")),
    )
)
(
    (area + line + rules)
    .properties(
        title="Best-so-far average precision by evaluation index",
        width=650,
        height=350,
    )
    .interactive()
)

alt.LayerChart(...)

## Best-so-far score by wall-clock time

Same cumulative maxima plotted against elapsed time. All arms ran
single-threaded on the same machine. The Optuna seeds are thin lines
behind the agent and PySR trajectories.

In [4]:
timed = best.assign(cumulative_min=lambda d: d["cumulative_s"] / 60)
time_lines = timed[timed["log"].str.startswith("optuna")].assign(label="optuna seed")
time_rest = pd.concat(
    [
        timed[timed["log"] == "agent"].assign(label="agent"),
        timed[timed["log"] == "pysr"].assign(label="pysr"),
    ]
)
seed_layer = (
    alt.Chart(time_lines)
    .mark_line(opacity=0.35, strokeWidth=1)
    .encode(
        x=alt.X("cumulative_min", title="Elapsed time (min)"),
        y=alt.Y("best_score", title="Best mean average precision so far"),
        detail="log",
        color=alt.value("#4c72b0"),
    )
)
rest_layer = (
    alt.Chart(time_rest)
    .mark_line()
    .encode(
        x="cumulative_min",
        y="best_score",
        color=alt.Color("label", title="arm"),
    )
)
time_rules = (
    alt.Chart(baselines)
    .mark_rule(strokeDash=[3, 3])
    .encode(y="mean_score", color=alt.Color("arm", title="reference"))
)
(
    (seed_layer + rest_layer + time_rules)
    .properties(
        title="Best-so-far average precision by elapsed time",
        width=650,
        height=350,
    )
    .interactive()
)

alt.LayerChart(...)

## Parameter trajectories

Chosen configurations over evaluations for the agent arm and the best
Optuna seed, defined as the seed with the highest final best score.
The regularization strength C is shown on a log10 scale.

In [5]:
final = optuna.sort_values("trial_index").groupby("log")["best_score"].last()
best_log = final.idxmax()
traj_logs = [best_log, "agent"]

branch = trials[trials["log"].isin(traj_logs)].copy()
branch["estimator"] = branch["params"].map(lambda p: p["estimator"])
branch["label"] = branch["log"].map({best_log: "optuna (best seed)", "agent": "agent"})
branch_chart = (
    alt.Chart(branch)
    .mark_point(filled=True, size=50)
    .encode(
        x=alt.X("trial_index", title="Evaluation index"),
        y=alt.Y("label", title=""),
        color=alt.Color("estimator", title="branch"),
        tooltip=["estimator", "trial_index"],
    )
    .properties(width=650, height=120, title="Estimator branch per evaluation")
)


def numeric_params(subset: pd.DataFrame, label: str) -> pd.DataFrame:
    rows = []
    for _, row in subset.iterrows():
        for key, value in row["params"].items():
            if isinstance(value, (bool, str)):
                continue
            name = "log10 C" if key == "C" else key
            shown = float(np.log10(value)) if key == "C" else value
            rows.append(
                {
                    "trial_index": row["trial_index"],
                    "param": name,
                    "value": shown,
                    "label": label,
                }
            )
    return pd.DataFrame(rows)


traj = pd.concat(
    [
        numeric_params(trials[trials["log"] == best_log], "optuna (best seed)"),
        numeric_params(trials[trials["log"] == "agent"], "agent"),
    ]
)
param_chart = (
    alt.Chart(traj)
    .mark_point(filled=True, size=40)
    .encode(
        x=alt.X("trial_index", title="Evaluation index"),
        y=alt.Y("value", title="Parameter value"),
        color=alt.Color("label", title="arm"),
        tooltip=["param", "value", "label"],
    )
    .facet(column="param", spacing=15)
    .resolve_scale(y="independent")
    .properties(title="Numeric parameter values per evaluation")
)
branch_chart & param_chart

alt.VConcatChart(...)

## Agent decision trail

Rationales are logged verbatim from the model output. Latency covers
one request round trip including reasoning tokens.

In [6]:
agent_rows = trials[trials["log"] == "agent"].sort_values("trial_index")
trail = pd.DataFrame(
    {
        "trial": agent_rows["trial_index"],
        "estimator": [p["estimator"] for p in agent_rows["params"]],
        "params": [
            json.dumps(
                {k: v for k, v in p.items() if k != "estimator"},
                separators=(",", ":"),
            )
            for p in agent_rows["params"]
        ],
        "ap": agent_rows["mean_score"].round(4),
        "latency_s": agent_rows["payload"].map(lambda p: p["latency_s"]).round(1),
        "rationale": [p["rationale"] for p in agent_rows["payload"]],
    }
)
trail.head(12)

,trial,estimator,params,ap,latency_s,rationale
0,0,random_forest,"{""n_estimators"":200,""max_depth"":8,""min_samples...",0.6341,16.4,"With no prior trials, start from a strong defa..."
1,1,random_forest,"{""n_estimators"":300,""max_depth"":14,""min_sample...",0.7676,28.6,"Building on the incumbent forest, I test deepe..."
2,2,random_forest,"{""n_estimators"":400,""max_depth"":20,""min_sample...",0.7785,19.2,The large jump from t00 to t01 came from deepe...
3,3,random_forest,"{""n_estimators"":400,""max_depth"":20,""min_sample...",0.8194,45.1,The incumbent t02 sits at the boundary on ever...
4,4,random_forest,"{""n_estimators"":400,""max_depth"":20,""min_sample...",0.7926,30.3,The best trial (t03) showed that unweighted cl...
5,5,logistic,"{""use_interactions"":true,""C"":1.0,""penalty"":""l1...",0.5972,48.0,The best random-forest config (t03) already si...
6,6,random_forest,"{""n_estimators"":400,""max_depth"":20,""min_sample...",0.8227,29.1,Incumbent t03 sits at the capacity boundaries ...
7,7,random_forest,"{""n_estimators"":400,""max_depth"":20,""min_sample...",0.8240,18.7,"Building on the best trial (t06), I continue t..."
8,8,random_forest,"{""n_estimators"":400,""max_depth"":20,""min_sample...",0.8221,18.3,Holding the best-performing settings (400 tree...
9,9,random_forest,"{""n_estimators"":400,""max_depth"":20,""min_sample...",0.8182,49.4,The strong region is clearly RF with 400 trees...


In [7]:
usage = pd.DataFrame(
    {
        "trial": agent_rows["trial_index"],
        "latency_s": agent_rows["payload"].map(lambda p: p["latency_s"]),
        "input": agent_rows["payload"]
        .map(lambda p: p["usage"]["input_tokens"])
        .cumsum(),
        "output": agent_rows["payload"]
        .map(lambda p: p["usage"]["output_tokens"])
        .cumsum(),
        "reasoning": agent_rows["payload"]
        .map(lambda p: p["usage"].get("output_reasoning_tokens", 0))
        .cumsum(),
    }
)
latency_chart = (
    alt.Chart(usage)
    .mark_bar()
    .encode(
        x=alt.X("trial", title="Evaluation index"),
        y=alt.Y("latency_s", title="Request latency (s)"),
    )
    .properties(width=300, height=250, title="Model latency per step")
)
tokens = usage.melt(
    id_vars="trial",
    value_vars=["input", "output", "reasoning"],
    var_name="kind",
    value_name="cumulative tokens",
)
token_chart = (
    alt.Chart(tokens)
    .mark_line()
    .encode(
        x=alt.X("trial", title="Evaluation index"),
        y=alt.Y("cumulative tokens", title="Cumulative tokens"),
        color=alt.Color("kind", title="token kind"),
    )
    .properties(width=300, height=250, title="Cumulative token usage")
)
latency_chart | token_chart

alt.HConcatChart(...)

## PySR expressions

Mean cross-validation and holdout average precision by expression
complexity, averaged over the folds in which each complexity was
present. Expressions operate on standardized features.

The ground-truth labeling rule is a disjunction of thresholded
products (torque times rotational speed, tool wear times torque,
temperature difference combined with rotational speed), so the
recoverable structure is rational in the standardized variables.

In [8]:
pysr_rows = trials[trials["log"] == "pysr"].copy()
pysr_rows["complexity"] = pysr_rows["params"].map(lambda p: p["complexity"])
pysr_rows["expression"] = pysr_rows["params"].map(lambda p: p["expression"])
pysr_rows["holdout_ap"] = pysr_rows["payload"].map(lambda p: p.get("holdout_ap"))
pareto = (
    pysr_rows.groupby("complexity")
    .agg(cv=("mean_score", "mean"), holdout=("holdout_ap", "mean"))
    .reset_index()
)
cv_pts = (
    alt.Chart(pareto)
    .mark_line(point=True)
    .encode(
        x=alt.X("complexity", title="Expression complexity"),
        y=alt.Y("cv", title="Average precision", scale=alt.Scale(zero=False)),
        color=alt.value("#4c72b0"),
    )
)
hold_pts = (
    alt.Chart(pareto)
    .mark_circle(size=60)
    .encode(
        x="complexity",
        y=alt.Y("holdout", title="Average precision (holdout)"),
        color=alt.value("#dd8452"),
    )
)
(
    (cv_pts + hold_pts)
    .properties(title="PySR score by expression complexity", width=650, height=320)
    .interactive()
)

alt.LayerChart(...)

In [9]:
top = (
    pysr_rows.sort_values("mean_score", ascending=False)
    .groupby("complexity", as_index=False)
    .agg(cv=("mean_score", "mean"), expression=("expression", "last"))
    .sort_values("cv", ascending=False)
    .head(8)
)
top.assign(cv=top["cv"].round(4))

,complexity,cv,expression
9,21,0.5455,"max(max(tool_wear, torque + max(tool_wear, max..."
10,23,0.5336,"max(max(torque, 0.026781622) * 0.3414368, max(..."
11,25,0.4965,max((torque + 0.19409576) * (torque * (0.06169...
7,17,0.4335,"max(min(0.867012, min((rpm * 0.03357151) * 0.9..."
6,15,0.3833,"min(0.9649074, max(max(0.35156637, max(tool_we..."
8,19,0.3663,"min(0.867012, max(rpm * min(torque * -0.576161..."
5,13,0.3663,"min(0.9649074, max(max(tool_wear, rpm * 0.3515..."
3,9,0.3210,"min(0.91264313, (rpm * rpm) * max(torque, 0.01..."


## Holdout summary

Final configurations were refit on the full search part and scored
once on the holdout. For PySR the holdout score is the mean over the
folds' hall-of-fame entries at the selected complexity. The rule
baseline omits the two stochastic failure modes, which bounds its
reachable average precision.

In [10]:
holdout.assign(
    cv_mean=holdout["cv_mean"].round(4),
    holdout_ap=holdout["holdout_ap"].round(4),
).sort_values("holdout_ap", ascending=False)[
    ["arm", "cv_mean", "holdout_ap", "folds_present"]
]

,arm,cv_mean,holdout_ap,folds_present
2,ground_truth_rules,NaN,0.8437,NaN
6,optuna_seed2,0.8258,0.8253,NaN
7,optuna_seed3,0.8242,0.8242,NaN
4,optuna_seed0,0.8265,0.8234,NaN
8,optuna_seed4,0.8306,0.8206,NaN
5,optuna_seed1,0.8157,0.8200,NaN
3,agent,0.8334,0.8166,NaN
1,baseline_default_forest,NaN,0.7692,NaN
9,pysr,0.5455,0.5622,1.0
0,baseline_default_logistic,NaN,0.5056,NaN


## Limitations

The PySR and agent arms are single runs at temperature 0; their
trajectories are case studies, not distributions. The Optuna arm
repeats five seeds to indicate run-to-run variance. Cross-validation
scores of the parametric arms share one split, so small differences
between arms should be read against the seed band. Reasoning tokens
of glm-5.3-flash are counted in the usage statistics but their
content is not logged.